### OpenMC simulation where we take the flux at each generation in a tally.

In [ ]:
import openmc
import matplotlib.pyplot as plt
import numpy as np
openmc.Materials.cross_sections = '/home/jonathon/openmc_xs/endfb-vii.1-hdf5/cross_sections.xml'

In [ ]:
"""Material setting up block"""
fuel = openmc.Material(material_id=1, name='fuel', temperature=909.04)
graphite = openmc.Material(material_id=2, name='graphite', temperature=909.04)
# Set number densities and densities.
fuel.add_nuclide(nuclide='U235', percent=7.010602E-05, percent_type='ao')
fuel.add_nuclide(nuclide='U238', percent=2.953509E-03, percent_type='ao')
fuel.add_nuclide(nuclide='Be9', percent=9.070844E-03, percent_type='ao')
fuel.add_nuclide(nuclide='F19', percent=4.837784E-02, percent_type='ao')
fuel.add_nuclide(nuclide='Li6', percent= 1.058009E-06, percent_type='ao')
fuel.add_nuclide(nuclide='Li7', percent=1.814063E-02, percent_type='ao')
# Graphite
graphite.add_nuclide(nuclide='C0', percent=1, percent_type='ao')
graphite.add_s_alpha_beta(name='c_Graphite', fraction=1.0)
# Densities of each material now
fuel.set_density(units='g/cm3', density=3.06)
graphite.set_density(units='g/cm3', density=1.843)
# Make an openmc.Materials() object and export it to xml
mats = openmc.Materials(materials=[fuel, graphite])
mats.export_to_xml()

In [ ]:
"""Geometry"""
# Planes for outer box
centroid = (0.0, 0.0, 0.0)
xPitch=5.08*2
yPitch=5.08*2
dz=20.0
xPlu_outer = openmc.XPlane(x0=centroid[0] + xPitch/2.0)
xNeg_outer = openmc.XPlane(x0=centroid[0] - xPitch/2.0)
yPlu_outer = openmc.YPlane(y0=centroid[1] + yPitch/2.0)
yNeg_outer = openmc.YPlane(y0=centroid[1] - yPitch/2.0)
zPlu = openmc.ZPlane(z0=centroid[2] + dz/2.0)
zNeg = openmc.ZPlane(z0=centroid[2] - dz/2.0)
# Planes for inner box
xPitch=4.75191*2
yPitch=4.75191*2
xPlu_inner = openmc.XPlane(x0=centroid[0] + xPitch/2.0)
xNeg_inner = openmc.XPlane(x0=centroid[0] - xPitch/2.0)
yPlu_inner = openmc.YPlane(y0=centroid[1] + yPitch/2.0)
yNeg_inner = openmc.YPlane(y0=centroid[1] - yPitch/2.0)

# Cylinder
cyl = openmc.ZCylinder(r=0.762)

# Boundary conditions
xPlu_outer.boundary_type = 'reflective'
xNeg_outer.boundary_type = 'reflective'
yPlu_outer.boundary_type = 'reflective'
yNeg_outer.boundary_type = 'reflective'
zPlu.boundary_type = 'reflective'
zNeg.boundary_type = 'reflective'

# Inside the outer box
outer_box_inside = -xPlu_outer & +xNeg_outer & +yNeg_outer & -yPlu_outer
# Between positive and negative zPlanes
between_the_z = -zPlu & +zNeg
# Inside the inner box
inner_box_inside = -xPlu_inner & +xNeg_inner & +yNeg_inner & -yPlu_inner


# The fuel region:
fuel_region = (outer_box_inside & ~inner_box_inside & between_the_z) | (-cyl & between_the_z)
# The graphite region
graphite_region = +cyl & inner_box_inside & between_the_z



# Fuel cell
fuel_cell = openmc.Cell(fill=fuel, region=fuel_region)
# Graphite cell
graphite_cell = openmc.Cell(fill=graphite, region=graphite_region)



# Universe
uni = openmc.Universe()
uni.add_cell(fuel_cell)
uni.add_cell(graphite_cell)


# OpenMC Geometry
geom = openmc.Geometry()
# Our geometry needs a root universe which is the top-level universe that all other universes live in.
geom.root_universe = uni
# Finally, we finalize the geometry by exporting the openmc.Geometry() object to an xml file:
geom.export_to_xml()

In [ ]:
"""Tallies"""
fuel_flux_tally = openmc.Tally(name="fuelFlux", tally_id=1)
fuel_flux_tally.scores = ['loc']
fuelFilter = openmc.CellFilter(bins=fuel_cell)
fuel_flux_tally.filters = [fuelFilter,]

tallies = openmc.Tallies(tallies=[fuel_flux_tally,])
tallies.export_to_xml()

In [ ]:
"""Starting source and settings"""
# Make a point source at the center of the problem
point = openmc.stats.Point((0.0,0.0,0.0))
# Define the starting source
source = openmc.IndependentSource(space=point)
settings = openmc.Settings()
settings.source = source
settings.batches = 300
settings.inactive = 10
settings.particles = 10000
settings.temperature['method'] = 'interpolation'
settings.export_to_xml()

In [ ]:
# Do the following to init() run() and finalize() results.
res = {}
import copy
import openmc.lib
openmc.lib.init()
openmc.lib.simulation_init()
for b in range(300):
  tally = openmc.lib.tallies[1]
  openmc.lib.next_batch()
  result = tally.results
  res[b] = copy.deepcopy(result)
openmc.lib.simulation_finalize()
openmc.lib.finalize()

In [ ]:
res

In [ ]:
56.92644368**2